[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garrygu/newegg-ai-workshop/blob/main/lv1-beginner-v2/archive/AI_Guessing_Game.ipynb)


# 🎮 Session 4 — Build an AI Guessing Game

Welcome! In this class, you will **put everything together** and build a small AI game.

You will use:
- **images** from earlier sessions
- an **image classifier** if you have one
- a **friendly game host**
- a **game loop** that keeps score

## Today's Goal
By the end of class, you will have a playable mini game:
1. Show an image
2. Guess the label
3. Let the AI guess too
4. Compare answers
5. Earn points!

---

## 🌟 Big Idea

A **game loop** means the game repeats the same steps again and again:

**show image → get guess → check answer → update score → next round**

That is how many games work!


## ✅ Before You Start

Please make sure:
- your images are inside the `game_assets/` folder
- image files look like `.png`, `.jpg`, or `.jpeg`
- if you have a trained model, place it at `models/classifier.pt`

### Helpful filename tip
It is easiest if your image filenames begin with the correct label.

Examples:
- `cat_01.png`
- `dog_02.jpg`
- `frog_green.png`

Then the notebook can figure out the correct answer from the filename.


In [ ]:
# Run this cell first
from pathlib import Path
import random
import math
import platform

import matplotlib.pyplot as plt
from PIL import Image

# Create folders if they do not exist yet
for folder in ["game_assets", "models"]:
    Path(folder).mkdir(exist_ok=True)

print("✅ Python version:", platform.python_version())
print("✅ Folders are ready.")
print("📁 Put your pictures in: game_assets/")
print("🧠 Put your model in: models/classifier.pt  (optional)")


## 1) Find Your Images

Let's look for pictures that the game can use.


In [ ]:
ASSET_DIR = Path("game_assets")
image_paths = sorted(
    [p for p in ASSET_DIR.glob("**/*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
)

print(f"Found {len(image_paths)} image(s).")

for p in image_paths[:10]:
    print(" -", p.name)

if len(image_paths) == 0:
    print("\n⚠️ No images found yet. Add some pictures to game_assets/ and run this cell again.")


## 2) Preview Some Images

This helps us check whether the image folder is correct.


In [ ]:
def show_image_grid(paths, n=6):
    if len(paths) == 0:
        print("No images to show yet.")
        return

    n = min(n, len(paths))
    cols = min(3, n)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, path in enumerate(paths[:n], start=1):
        img = Image.open(path).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(path.name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_image_grid(image_paths, n=6)


## 3) Get the Correct Label from the Filename

For this beginner game, we will use the filename to figure out the correct answer.

### Rule
The notebook will read the part **before the first underscore**.

Examples:
- `cat_01.png` → `cat`
- `dog_happy.jpg` → `dog`
- `frog.png` → `frog`

This keeps the game simple and beginner-friendly.


In [ ]:
def label_from_filename(path):
    stem = path.stem.lower().strip()
    if "_" in stem:
        return stem.split("_")[0]
    return stem

# Try it on a few files
for p in image_paths[:5]:
    print(p.name, "→", label_from_filename(p))


## 4) Optional: Load Your AI Classifier

If you trained a model earlier, you can load it here.

If no model is found, the game will still work in **easy mode**.


In [ ]:
# Optional model loading
import torch
import torch.nn as nn
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# IMPORTANT:
# Edit this list to match the labels your model learned.
classes = ["cat", "dog", "frog", "car", "bird"]

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

MODEL_PATH = Path("models/classifier.pt")
model = None

if MODEL_PATH.exists():
    try:
        model = SimpleCNN(num_classes=len(classes)).to(device)
        state = torch.load(MODEL_PATH, map_location=device)
        model.load_state_dict(state)
        model.eval()
        print("✅ Model loaded from", MODEL_PATH)
    except Exception as e:
        print("⚠️ Model found, but it could not be loaded.")
        print("Reason:", e)
        print("The game can still run without the model.")
else:
    print("ℹ️ No model found. The game will run without AI prediction.")


## 5) Let the AI Make a Prediction

This function lets the model guess the image label.

If there is no model, it will simply say `"unknown"`.


In [ ]:
def predict_with_ai(img_path):
    if model is None:
        return "unknown", 0.0

    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = probs.max(dim=1)

    label = classes[predicted.item()]
    conf = float(confidence.item())
    return label, conf

if len(image_paths) > 0:
    test_label, test_conf = predict_with_ai(image_paths[0])
    print("AI guess:", test_label)
    print("Confidence:", round(test_conf, 3))


## 6) Build a Friendly Game Host

The game host gives fun messages to the player.

This is simpler and faster than a big chatbot, which is great for middle school students.


In [ ]:
def game_host_message(is_correct, player_guess, true_label, ai_guess):
    happy_messages = [
        "🎉 Nice job! You got it right!",
        "🌟 Awesome! Great guessing!",
        "😄 Correct! You are doing great!",
        "🚀 You got it! Keep going!"
    ]

    try_again_messages = [
        f"🙂 Good try! The correct answer was {true_label}.",
        f"🧠 Nice effort! This one was {true_label}.",
        f"👍 Almost! The answer was {true_label}.",
        f"🎯 Keep practicing! This image was {true_label}."
    ]

    if is_correct:
        return random.choice(happy_messages)
    else:
        if ai_guess != "unknown":
            return random.choice(try_again_messages) + f" The AI guessed: {ai_guess}."
        return random.choice(try_again_messages)

print(game_host_message(True, "cat", "cat", "cat"))
print(game_host_message(False, "dog", "cat", "cat"))


## 7) One Round of the Game

In one round:
1. show one image
2. ask for the player's guess
3. check the true label
4. let the AI guess too
5. return the points

### Scoring
- **+10** if the player is correct
- **0** if the player is incorrect

Simple scoring makes the game easy to understand.


In [ ]:
def play_one_round(img_path):
    true_label = label_from_filename(img_path)

    # Show image
    img = Image.open(img_path).convert("RGB")
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # Ask player
    player_guess = input("What do you think this is? ").strip().lower()

    # AI guess
    ai_guess, ai_conf = predict_with_ai(img_path)

    # Check answer
    is_correct = (player_guess == true_label)
    points = 10 if is_correct else 0

    print("\n--- Round Result ---")
    print("✅ Correct label :", true_label)
    print("🙋 Your guess    :", player_guess)
    print("🤖 AI guess      :", ai_guess)
    if ai_guess != "unknown":
        print("📊 AI confidence :", round(ai_conf * 100, 1), "%")

    print(game_host_message(is_correct, player_guess, true_label, ai_guess))
    print("🏅 Points earned :", points)

    return points

# Test one round if you have images:
# play_one_round(image_paths[0])


## 8) Full Game Loop

Now we repeat the round several times.

That is the **game loop**!


In [ ]:
def play_game(rounds=3):
    if len(image_paths) == 0:
        print("⚠️ Please add images to game_assets/ first.")
        return

    rounds = min(rounds, len(image_paths))
    chosen_images = random.sample(image_paths, rounds)

    score = 0

    print("🎮 Starting the AI Guessing Game!")
    print(f"We will play {rounds} round(s).")

    for round_number, img_path in enumerate(chosen_images, start=1):
        print("\n" + "=" * 40)
        print(f"Round {round_number}/{rounds}")
        score += play_one_round(img_path)
        print(f"⭐ Total score so far: {score}")

    print("\n" + "=" * 40)
    print("🏁 Game Over!")
    print("Final score:", score)

    if score == rounds * 10:
        print("🌈 Perfect score! Amazing work!")
    elif score >= rounds * 5:
        print("👏 Great job! You did really well.")
    else:
        print("😊 Nice try! Keep practicing and have fun.")


## 9) Play the Game

Change the number below if you want more or fewer rounds.


In [ ]:
play_game(rounds=3)


## 10) Make It Your Own ✨

Try changing one thing at a time:

### Easy ideas
- change the score system
- change the game host messages
- play 5 rounds instead of 3
- add new pictures
- add more labels to `classes`

### Challenge ideas
- subtract points for wrong answers
- keep track of wins and losses
- show the AI guess only after the player answers
- create a themed version:
  - animal game
  - car game
  - superhero game
  - food game

That is how coders improve projects: **test, change, and try again**.


## 🏆 Wrap-Up

Today you built a real mini AI project with:
- images
- labels
- optional AI prediction
- a game loop
- score tracking

That is a big step from “using AI” to **building with AI**.

Great work!
